# P1 additional diagnostic studies and compiler compatibility

Use the existing **P1 native kernel** on Wahab. Upload this single notebook into `~/P1`; all source and fixtures are embedded. This notebook does not rerun Q12/Q13/Q14/Q16 or the two studies in the earlier pending-experiments notebook.

**Default: smoke only, eight workers.** Both new smoke families run before either full diagnostic family. Full diagnostics add 176,640 native executions: causal controls (61,440) and one-factor sensitivity (115,200). These fixed sample sizes are diagnostic, not a power guarantee for small effects. No policy is learned or tuned here.

The compiler check is translation/parser compatibility on exact upstream fixtures. It does not implement the full upstream application comparison or TUF. A genuinely adaptive entangled application also remains to be developed; see the included plan.

Local verification: eleven new engineering tests and eleven retained tests pass. **No native smoke/full execution has been performed locally.**


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.home() / "P1"
RUN_ROOT = PROJECT_DIR / "additional-diagnostics-20261024-v1"
RESULTS = RUN_ROOT / "results"
WORKERS = 8
MODE = "smoke"                  # "smoke" first; then "full" to collect diagnostics
FAMILIES = ("causal", "robustness")
QOALA_REPO = None                # Optional explicit path to the existing pinned Qoala repo
PYTHON_OVERRIDE = None           # Optional path to the existing native Python
COMPILER_BIN_DIR = None          # Optional installed qoala-mlir bin directory

assert MODE in ("smoke", "full")
assert 1 <= WORKERS <= 8
assert FAMILIES and len(FAMILIES) == len(set(FAMILIES))
assert all(f in ("causal", "robustness") for f in FAMILIES)
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT_DIR)
print("Run:", RUN_ROOT)
print("Mode:", MODE, "families:", FAMILIES, "workers:", WORKERS)
print("Run study families sequentially; do not launch a second eight-worker notebook in this allocation.")


In [ ]:
import base64, hashlib, io, json, os, signal, subprocess, zipfile
PAYLOAD_SHA256 = 'af77087906d242a1cb94c563db71d27d02789a7178ce4462f3368fac8f5cb412'
PAYLOAD = '...'
blob = base64.b64decode(PAYLOAD)
assert hashlib.sha256(blob).hexdigest() == PAYLOAD_SHA256
RUN_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for entry in archive.infolist():
        p = Path(entry.filename)
        if p.is_absolute() or ".." in p.parts:
            raise RuntimeError("Unsafe package path")
        target = RUN_ROOT / p
        if target.exists() and target.is_file() and target.read_bytes() != archive.read(entry):
            raise RuntimeError(f"Existing source differs; preserve it and choose a new RUN_ROOT: {target}")
    archive.extractall(RUN_ROOT)
PACKAGE = RUN_ROOT / "P1_Additional_Studies"
SOURCE = PACKAGE / "source"
RESULTS.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / "PACKAGE_IDENTITY.json").write_text(json.dumps({"sha256": PAYLOAD_SHA256, "source_archive": "P1_Additional_Studies_Source.zip"}, indent=2)+"\n")
print("Verified package:", PACKAGE)


Read the full plan before interpreting results. E1/E2 are diagnostic additions. E4 (adaptive entangled application), E5 (full compiler/TUF comparison), and any service-preserving confirmation are explicitly unfinished tasks, not experiments this notebook silently implements.

In [ ]:
from IPython.display import Markdown, display
display(Markdown("```text\n" + (PACKAGE / "README.txt").read_text() + "\n```"))


In [ ]:
def run_command(command, allow_failure=False):
    """Stream logs; interrupt the process group, including spawned workers."""
    print("Executing:", " ".join(map(str, command)), flush=True)
    with (RUN_ROOT / "LAUNCH.log").open("a", buffering=1) as log:
        log.write("\nCOMMAND: " + " ".join(map(str, command)) + "\n")
        process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True); log.write(line)
            status = process.wait()
        except BaseException:
            try: os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError: pass
            try: process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                try: os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError: pass
            raise
    if status and not allow_failure:
        raise RuntimeError(f"Command failed ({status}). Preserve logs/raw files; do not report effect estimates from incomplete studies.")
    return status

run_command([sys.executable, SOURCE / "test_suite.py"])
run_command([sys.executable, PACKAGE / "P1_Feedback_Closure/source/offline_tests.py"])


The upstream compiler gate is independent of the diagnostic simulator runs. Missing tools produce an explicit blocked report. Do not install a compiler or upgrade the working P1 runtime in this notebook. For an actual compiler benchmark, use a separately pinned toolchain and complete semantic validation.

In [ ]:
probe = """import importlib, json
names=('numpy','netsquid','pydynaa','qoala','netqasm')
found={}; errors={}
for name in names:
    try:
        m=importlib.import_module(name)
        found[name]=dict(path=str(getattr(m,'__file__','')),version=str(getattr(m,'__version__','unreported')))
    except Exception as e: errors[name]=repr(e)
print(json.dumps(dict(found=found,errors=errors)))
"""
candidates = ([Path(PYTHON_OVERRIDE)] if PYTHON_OVERRIDE else
              [PROJECT_DIR / ".venv-p1/bin/python", PROJECT_DIR / ".venv/bin/python", Path(sys.executable)])
probes = []; NATIVE_PYTHON = None
for candidate in dict.fromkeys(candidates):
    if not candidate.is_file(): continue
    attempt = subprocess.run([str(candidate), "-c", probe], capture_output=True, text=True, timeout=60)
    try: result = json.loads(attempt.stdout.strip().splitlines()[-1])
    except Exception: result = {"errors": {"probe": attempt.stderr or attempt.stdout}}
    probes.append({"python": str(candidate), **result})
    if attempt.returncode == 0 and not result["errors"]:
        NATIVE_PYTHON = str(candidate); break
(RUN_ROOT / "ENVIRONMENT_PROBE.json").write_text(json.dumps(probes, indent=2)+"\n")
if NATIVE_PYTHON is None:
    raise RuntimeError("Native P1 dependencies are unavailable. Select the existing NetSquid/Qoala P1 kernel or set PYTHON_OVERRIDE. No native simulation has been launched.")
print("Native Python:", NATIVE_PYTHON)


In [ ]:
compiler_cmd = [NATIVE_PYTHON, SOURCE / "compiler_gate.py", "--folder", RESULTS / "compiler-gate"]
if COMPILER_BIN_DIR is not None:
    compiler_cmd += ["--tools-dir", COMPILER_BIN_DIR]
run_command(compiler_cmd, allow_failure=True)
compiler_report = json.loads((RESULTS / "compiler-gate/COMPILER_GATE.json").read_text())
print("Compiler gate:", compiler_report["status"])
print("Full upstream performance comparison: NOT IMPLEMENTED by this gate.")


In [ ]:
# Run BOTH smoke families before starting either full diagnostic family.
# Resume requires exact source/protocol identity. Full N cannot be changed here.
def command(family, mode):
    cmd = [NATIVE_PYTHON, SOURCE / "study_runner.py", "run", "--family", family,
           "--mode", mode, "--folder", RESULTS / (family + "-" + mode), "--workers", WORKERS]
    if QOALA_REPO is not None: cmd += ["--repo", QOALA_REPO]
    return cmd

def cost_estimate(family):
    records = [json.loads(p.read_text()) for p in (RESULTS / (family+"-smoke") / "checkpoints").glob("*.json")]
    n = 512 if family == "causal" else 256
    if not records: return
    seconds = sum(r["wall_seconds"] for r in records)/len(records)*n
    size = sum((RESULTS / (family+"-smoke") / r["raw"]).stat().st_size for r in records)/len(records)*n
    print(f"{family}: rough serial CPU {seconds/3600:.2f} h; idealized /{WORKERS} wall {seconds/3600/WORKERS:.2f} h; compressed raw {size/2**30:.2f} GiB.")
    print("Smoke extrapolation only: scheduler contention, heterogeneity and filesystem costs can change these estimates.")

try:
    for family in FAMILIES:
        run_command(command(family, "smoke")); cost_estimate(family)
    if MODE == "full":
        for family in FAMILIES:
            run_command(command(family, "full"))
    print("Finished requested diagnostic mode. Independent replay of saved native raw data remains required.")
finally:
    run_command([NATIVE_PYTHON, SOURCE / "study_runner.py", "export", "--folder", RESULTS], allow_failure=True)


After smoke succeeds, change only `MODE` to `"full"` and run the launch cell again (or Run All). The source and fixed sample sizes stay unchanged. Run the existing pending full studies separately within the same eight-worker allocation. A null result or a wide interval is an outcome, not a reason to extend the sample size.

After a full run, retain the raw directory on the server. Run the saved-raw replay commands printed below before making paper claims. Upload the compact review ZIP for inspection; it includes summaries, protocols, logs and hashes, while raw/failed traces stay on the server.

In [ ]:
from IPython.display import FileLink
review = RESULTS.parent / (RESULTS.name + "-compact-review.zip")
if review.exists():
    display(FileLink(os.path.relpath(review, Path.cwd())))
else:
    print("No review archive yet; complete preflight and the requested mode first.")
for family in FAMILIES:
    full = RESULTS / (family+"-full")
    print("Replay:", NATIVE_PYTHON, SOURCE / "study_runner.py", "replay --folder", full)
print("Plan:", PACKAGE / "README.txt")
